# Supervised Fine-Tuning with SFTTrainer

This notebook demonstrates how to fine-tune the `HuggingFaceTB/SmolLM2-135M` model using the `SFTTrainer` from the `trl` library. The notebook cells run and will finetune the model. You can select your difficulty by trying out different datasets.

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Exercise: Fine-Tuning SmolLM2 with SFTTrainer</h2>
    <p>Take a dataset from the Hugging Face hub and finetune a model on it. </p> 
    <p><b>Difficulty Levels</b></p>
    <p>🐢 Use the `HuggingFaceTB/smoltalk` dataset</p>
    <p>🐕 Try out the `bigcode/the-stack-smol` dataset and finetune a code generation model on a specific subset `data/python`.</p>
    <p>🦁 Select a dataset that relates to a real world use case your interested in</p>
</div>

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
# Install the requirements in Google Colab
# !pip install transformers datasets trl huggingface_hub

# Authenticate to Hugging Face

from huggingface_hub import login
HUGGINGFACE_TOKEN = "HUGGINGFACE_TOKEN"
login(token=HUGGINGFACE_TOKEN)

# for convenience you can create an environment variable containing your hub token as HF_TOKEN

In [3]:
# Import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-135M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
).to(device)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Set up the chat format
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

# Set our name for the finetune to be saved &/ uploaded to
finetune_name = "SmolLM2-FT-MyDataset_2"
finetune_tags = ["smol-course", "module_1"]

# Generate with the base model

Here we will try out the base model which does not have a chat template. 

In [4]:
# Let's test the base model before training
prompt = "Write a haiku about programming"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=100)
print("Before training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Before training:
user
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a haiku about programming
Write a


## Dataset Preparation

We will load a sample dataset and format it for training. The dataset should be structured with input-output pairs, where each input is a prompt and the output is the expected response from the model.

**TRL will format input messages based on the model's chat templates.** They need to be represented as a list of dictionaries with the keys: `role` and `content`,.

In [5]:
# Load a sample dataset
from datasets import load_dataset

# TODO: define your dataset and config using the path and name parameters
ds = load_dataset(path="davanstrien/haiku_dpo", name="default")


README.md:   0%|          | 0.00/13.4k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/3.63M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4123 [00:00<?, ? examples/s]

In [6]:
print(ds)

DatasetDict({
    train: Dataset({
        features: ['question', 'generation_model', 'generation_prompt', 'generations', 'scores', 'chosen', 'chosen_score', 'rejected', 'rejected_score', 'tie', 'difference_in_score', 'system'],
        num_rows: 4123
    })
})


In [ ]:
# Split the dataset into train and test
ds = ds["train"].train_test_split(test_size=0.1)


In [9]:
# Check the dataset
print("Question:", ds["test"][0]["question"])
print("Answer:", ds["test"][0]["chosen"])

Question: Compose a haiku that describes the tarantula's hunting skills.
Answer: Silent hunter waits,
Eight legs ready to entwine,
Tarantula's prey.


In [12]:
print(ds["train"][0])

{'question': "Can you compose a haiku about a sparrow's flight?", 'generation_model': ['TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ'], 'generation_prompt': ["<|im_start|>system\nYou are a poet specialising i

In [10]:
def process_dataset(sample):
    question, answer = sample["question"], sample["chosen"]
    messages = [{"role": "user", "content": question}, {"role": "assistant", "content": answer}]
    
    return {"messages": messages}


ds = ds.map(process_dataset)

Map:   0%|          | 0/3710 [00:00<?, ? examples/s]

Map:   0%|          | 0/413 [00:00<?, ? examples/s]

In [11]:
print(ds["train"][0])

{'question': "Can you compose a haiku about a sparrow's flight?", 'generation_model': ['TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ', 'TheBloke/OpenHermes-2.5-Mistral-7B-AWQ'], 'generation_prompt': ["<|im_start|>system\nYou are a poet specialising i

## Configuring the SFTTrainer

The `SFTTrainer` is configured with various parameters that control the training process. These include the number of training steps, batch size, learning rate, and evaluation strategy. Adjust these parameters based on your specific requirements and computational resources.

In [15]:
def formatting_func(example):
    """
    messages 리스트를 하나의 text 문자열로 변환
    """
    text = ""
    for msg in example["messages"]:
        role = msg["role"]
        content = msg["content"]
        text += f"{role}: {content}\n"  # "user: 질문" 형태로 변환
    return text.strip()  # 문자열(str) 직접 반환


# Configure the SFTTrainer
sft_config = SFTConfig(
    output_dir="./sft_output_2",
    max_steps=1000,  # Adjust based on dataset size and desired training duration
    per_device_train_batch_size=4,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=10,  # Frequency of logging training metrics
    save_steps=100,  # Frequency of saving model checkpoints
    eval_strategy="steps",  # Evaluate the model at regular intervals
    eval_steps=50,  # Frequency of evaluation
    use_mps_device=False,
    hub_model_id=finetune_name,  # Set a unique name for your model
    
)


trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    tokenizer=tokenizer,
    formatting_func=formatting_func
)

# TODO: 🦁 🐕 align the SFTTrainer params with your chosen dataset. For example, if you are using the `bigcode/the-stack-smol` dataset, you will need to choose the `content` column`

/tmp/ipykernel_27984/1594447740.py:29: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


Applying formatting function to train dataset:   0%|          | 0/3710 [00:00<?, ? examples/s]

Converting train dataset to ChatML:   0%|          | 0/3710 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/3710 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3710 [00:00<?, ? examples/s]

...

Converting eval dataset to ChatML:   0%|          | 0/413 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/413 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/413 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/413 [00:00<?, ? examples/s]

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## Training the Model

With the trainer configured, we can now proceed to train the model. The training process will involve iterating over the dataset, computing the loss, and updating the model's parameters to minimize this loss.

In [16]:
import os
import torch

# 기존 GPU 설정 해제
torch.cuda.empty_cache()

# 특정 GPU 사용하도록 강제 설정
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# 다시 CUDA 장치 확인
torch.cuda.set_device(0)
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(torch.cuda.current_device()))


0
NVIDIA A100-SXM4-80GB


In [17]:
# Train the model
trainer.train()

# Save the model
trainer.save_model(f"./{finetune_name}")

Step,Training Loss,Validation Loss
50,1.885900,1.881440
100,1.697600,1.767852
150,1.842400,1.718782
200,1.642400,1.679439
250,1.621300,1.646525
300,1.623700,1.622342
350,1.559800,1.601143
400,1.538400,1.575473
450,1.669600,1.573827
500,1.423000,1.544471


In [18]:
trainer.push_to_hub(tags=finetune_tags)

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

training_args.bin:   0%|          | 0.00/5.56k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/lovesally/SmolLM2-FT-MyDataset_2/commit/638119b750446155a37fed2ff54109051d5965d1', commit_message='End of training', commit_description='', oid='638119b750446155a37fed2ff54109051d5965d1', pr_url=None, repo_url=RepoUrl('https://huggingface.co/lovesally/SmolLM2-FT-MyDataset_2', endpoint='https://huggingface.co', repo_type='model', repo_id='lovesally/SmolLM2-FT-MyDataset_2'), pr_revision=None, pr_num=None)

<div style='background-color: lightblue; padding: 10px; border-radius: 5px; margin-bottom: 20px; color:black'>
    <h2 style='margin: 0;color:blue'>Bonus Exercise: Generate with fine-tuned model</h2>
    <p>🐕 Use the fine-tuned to model generate a response, just like with the base example..</p>
</div>

In [28]:
# Test the fine-tuned model on the same prompt

# Let's test the base model before training
prompt = "Write a short haiku about coding"

# Format with template
messages = [{"role": "user", "content": prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False)

# Generate response
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)

outputs = model.generate(**inputs, max_new_tokens=50,
                        eos_token_id=tokenizer.eos_token_id,
)
print("After training:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

After training:
user
Write a short haiku about coding
In the darkened night,
A code whispers to the soul,
Silent, yet alive.

Coding, a secret
In the darkened night,
Silent, alive,
Code whispers to the soul.




## 💐 You're done!

This notebook provided a step-by-step guide to fine-tuning the `HuggingFaceTB/SmolLM2-135M` model using the `SFTTrainer`. By following these steps, you can adapt the model to perform specific tasks more effectively. If you want to carry on working on this course, here are steps you could try out:

- Try this notebook on a harder difficulty
- Review a colleagues PR
- Improve the course material via an Issue or PR.